# Python para redes neuronales

# Importar librerias

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from keras.layers import Input, Flatten, Dense, Conv2D
from keras.models import Model
from keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

from keras.datasets import cifar10

2025-01-24 01:17:20.297770: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-01-24 01:17:20.424902: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-01-24 01:17:20.459933: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-01-24 01:17:20.707521: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-01-24 01:17:22.547309: W tensorflow/compiler/tf2

# Colectar los datos

In [2]:
NUM_CLASSES = 10

In [14]:
#(x_train, y_train), (x_test, y_test) = cifar10.load_data()
import pickle
import os

def load_cifar10_batch(file):
    with open(file, 'rb') as f:
        dict = pickle.load(f, encoding='bytes')
        X = dict[b'data']
        Y = dict[b'labels']
        X = X.reshape(len(X), 3, 32, 32).transpose(0, 2, 3, 1)
        Y = np.array(Y)
        return X, Y

def load_cifar10(data_dir):
    x_train = []
    y_train = []
    # Cargar los lotes de datos de entrenamiento
    for i in range(1, 6):
        file = os.path.join(data_dir, f'data_batch_{i}')
        X, Y = load_cifar10_batch(file)
        x_train.append(X)
        y_train.append(Y)
    x_train = np.concatenate(x_train)
    y_train = np.concatenate(y_train)
    
    # Cargar los datos de prueba
    x_test, y_test = load_cifar10_batch(os.path.join(data_dir, 'test_batch'))
    
    return (x_train, y_train), (x_test, y_test)


In [15]:
cifar10_path = '/home/edgarrt/Files/Datasets/cifar-10-batches-py/'
(x_train, y_train), (x_test, y_test) = load_cifar10(cifar10_path)

print("Datos de entrenamiento:", x_train.shape, y_train.shape)
print("Datos de prueba:", x_test.shape, y_test.shape)

Datos de entrenamiento: (50000, 32, 32, 3) (50000,)
Datos de prueba: (10000, 32, 32, 3) (10000,)


In [17]:
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

y_train = to_categorical(x_train, NUM_CLASSES)
y_test = to_categorical(y_test, NUM_CLASSES)

In [18]:
x_train[54, 12, 13, 1] 

0.0014455979

In [19]:
x_train.shape

(50000, 32, 32, 3)

# arquitectura de la red

In [20]:
input_layer = Input((32,32,3))

# Flaten convierte matriz a vector
x = Flatten()(input_layer)

#Dense capa de perceptron, solo del tip vector, noo acepta matriz
# la funcion de activacion permite a la neurona usar valores entre 0 y 1 (infinitos)
x = Dense(200, activation = 'relu')(x)
x = Dense(150, activation = 'relu')(x)

# necesito 10 neuronas porque tengo 10 clases, sofmax es multiclase a diferencia de sigmoid
output_layer = Dense(NUM_CLASSES, activation = 'softmax')(x)

model = Model(input_layer, output_layer)

In [21]:
model.summary()
# input y flaten no tienen parametros ya que solo tienen la entrada (no modifican nada)

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 3072)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 200)            │       614,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 150)            │        30,150 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 10)             │         1,510 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 646,260 (2.47 MB)

 Trainable params: 646,260 (2.47 MB)

 Non-trainable params: 0 (0.00 B)

# Entrenamiento

In [22]:
#Adam optimizadr
opt = Adam(learning_rate=0.0005)
model.compile(loss='categorical_crossentropy', optimizer=opt, metrics=['accuracy'])

In [ ]:
model.fit(x_train
          , y_train
          , batch_size=32
          , epochs=10
          , shuffle=True)

# Analisis

In [ ]:
model.evaluate(x_test, y_test)

In [ ]:
CLASSES = np.array(['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck'])

preds = model.predict(x_test)
preds_single = CLASSES[np.argmax(preds, axis = -1)]
actual_single = CLASSES[np.argmax(y_test, axis = -1)]

In [ ]:

n_to_show = 10
indices = np.random.choice(range(len(x_test)), n_to_show)

fig = plt.figure(figsize=(15, 3))
fig.subplots_adjust(hspace=0.4, wspace=0.4)

for i, idx in enumerate(indices):
    img = x_test[idx]
    ax = fig.add_subplot(1, n_to_show, i+1)
    ax.axis('off')
    ax.text(0.5, -0.35, 'pred = ' + str(preds_single[idx]), fontsize=10, ha='center', transform=ax.transAxes) 
    ax.text(0.5, -0.7, 'act = ' + str(actual_single[idx]), fontsize=10, ha='center', transform=ax.transAxes)
    ax.imshow(img)
